# 📊 Análisis de Calidad del Texto (CDESCR)

Este notebook analiza la calidad del texto en Complaints comparando:
1. Dataset original filtrado (antes del downsampling)
2. Dataset downsampled (25%)

## Métricas que mediremos:
- Longitud promedio (caracteres y palabras)
- Distribución de longitudes (percentiles)
- Vocabulario único
- Longitud de palabras promedio
- Palabras más frecuentes
- **Error porcentual** en cada métrica


In [6]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
from collections import Counter

# Configuración de rutas
BASE_DIR = Path(".." if Path.cwd().name == "notebooks" else ".")

print(f"📁 Working directory: {Path.cwd()}")
print(f"📁 Base directory: {BASE_DIR}")


📁 Working directory: c:\Users\moral\tec_final\notebooks
📁 Base directory: ..


In [7]:
# PASO 1: Cargar datasets
print("="*70)
print("CARGANDO DATASETS")
print("="*70)

# Dataset original
df_original = pd.read_parquet(BASE_DIR / "data/processed/complaints.parquet")
print(f"✅ Dataset original cargado: {len(df_original):,} registros")

# Dataset downsampled (25%)
df_downsampled = pd.read_parquet(BASE_DIR / "data/processed/complaints_filtered_downsampled.parquet")
print(f"✅ Dataset downsampled cargado: {len(df_downsampled):,} registros")


CARGANDO DATASETS
✅ Dataset original cargado: 2,137,711 registros
✅ Dataset downsampled cargado: 512,725 registros


In [8]:
# PASO 2: Aplicar los mismos filtros que usa el notebook 1.2.1
print("\n" + "="*70)
print("APLICANDO FILTROS (igual que 1.2.1)")
print("="*70)

# Verificar que df_original existe (ejecuta la celda 2 primero)
if 'df_original' not in globals():
    print("❌ ERROR: Ejecuta primero la celda 2 (Carga de datos)")
    raise NameError("df_original no está definido. Ejecuta la celda 2 primero.")

# Filtros:
# 1. Solo PROD_TYPE = 'V' o 'T'
df_filtered = df_original[df_original['PROD_TYPE'].isin(['V', 'T'])].copy()
print(f"[1] Después de filtrar PROD_TYPE: {len(df_filtered):,} registros")

# 2. Excluir YEARTXT = 9999
df_filtered = df_filtered[df_filtered['YEARTXT'] != 9999].copy()
print(f"[2] Después de excluir 9999: {len(df_filtered):,} registros")

# 3. Filtrar componentes válidos
df_filtered = df_filtered[
    (df_filtered['PROD_TYPE'] == 'V') | 
    (df_filtered['COMPDESC'].notna()) | 
    (df_filtered['COMPDESC'] == 'NONE')
].copy()
print(f"[3] Después de filtrar componentes: {len(df_filtered):,} registros")

print(f"\n📊 Dataset original filtrado listo: {len(df_filtered):,} registros")



APLICANDO FILTROS (igual que 1.2.1)
[1] Después de filtrar PROD_TYPE: 2,107,703 registros
[2] Después de excluir 9999: 2,053,391 registros
[3] Después de filtrar componentes: 2,053,391 registros

📊 Dataset original filtrado listo: 2,053,391 registros


In [9]:
# PASO 3: Análisis de calidad del texto (CDESCR)
print("\n" + "="*70)
print("ANÁLISIS DE CALIDAD DEL TEXTO")
print("="*70)

# Verificar que las variables necesarias existen
required_vars = ['df_filtered', 'df_downsampled']
missing = [v for v in required_vars if v not in globals()]
if missing:
    print(f"❌ ERROR: Faltan variables: {missing}")
    print("   Ejecuta las celdas anteriores en orden (1, 2, 3)")
    raise NameError(f"Faltan variables: {missing}")

def analizar_texto(df, nombre="Dataset"):
    """Analiza estadísticas del texto CDESCR"""
    print(f"\n📝 Analizando: {nombre}")
    
    # Filtrar registros con texto válido
    df_text = df[df['CDESCR'].notna() & (df['CDESCR'] != '')].copy()
    print(f"  ✅ Registros con texto: {len(df_text):,}")
    
    if len(df_text) == 0:
        return None
    
    # Longitud en caracteres
    df_text['length_chars'] = df_text['CDESCR'].str.len()
    
    # Longitud en palabras (aproximado)
    df_text['length_words'] = df_text['CDESCR'].str.split().str.len()
    
    # Estadísticas
    stats = {
        'n_registros': len(df_text),
        'promedio_chars': df_text['length_chars'].mean(),
        'mediana_chars': df_text['length_chars'].median(),
        'std_chars': df_text['length_chars'].std(),
        'p25_chars': df_text['length_chars'].quantile(0.25),
        'p75_chars': df_text['length_chars'].quantile(0.75),
        'p95_chars': df_text['length_chars'].quantile(0.95),
        'promedio_palabras': df_text['length_words'].mean(),
        'mediana_palabras': df_text['length_words'].median(),
    }
    
    # Vocabulario único
    todas_palabras = []
    for text in df_text['CDESCR'].fillna(''):
        palabras = re.findall(r'\b[a-zA-Z]+\b', text.lower())
        todas_palabras.extend(palabras)
    
    stats['vocabulario_unico'] = len(set(todas_palabras))
    stats['total_palabras'] = len(todas_palabras)
    
    # Longitud promedio de palabra
    if todas_palabras:
        stats['longitud_palabra_promedio'] = np.mean([len(p) for p in todas_palabras])
    else:
        stats['longitud_palabra_promedio'] = 0
    
    # Palabras más frecuentes
    word_freq = Counter(todas_palabras)
    stats['palabras_mas_frecuentes'] = word_freq.most_common(10)
    
    return stats

# Analizar ambos datasets
stats_original = analizar_texto(df_filtered, "Original (Filtrado)")
stats_downsampled = analizar_texto(df_downsampled, "Downsampled (25%)")



ANÁLISIS DE CALIDAD DEL TEXTO

📝 Analizando: Original (Filtrado)
  ✅ Registros con texto: 2,053,294

📝 Analizando: Downsampled (25%)
  ✅ Registros con texto: 512,706


In [10]:
# PASO 4: Comparar estadísticas y calcular errores
print("\n" + "="*70)
print("COMPARACIÓN Y CÁLCULO DE ERRORES")
print("="*70)

if stats_original and stats_downsampled:
    metricas = [
        'promedio_chars', 'mediana_chars', 'std_chars',
        'p25_chars', 'p75_chars', 'p95_chars',
        'promedio_palabras', 'mediana_palabras',
        'longitud_palabra_promedio'
    ]
    
    print("\n📊 COMPARACIÓN DE MÉTRICAS:")
    print("-" * 70)
    print(f"{'Métrica':<35} {'Original':>15} {'Downsampled':>15} {'Error %':>10}")
    print("-" * 70)
    
    errores = []
    for metrica in metricas:
        orig = stats_original.get(metrica, 0)
        down = stats_downsampled.get(metrica, 0)
        
        if orig != 0:
            error_pct = abs((orig - down) / orig) * 100
        else:
            error_pct = 0
        
        errores.append(error_pct)
        
        print(f"{metrica:<35} {orig:>15.2f} {down:>15.2f} {error_pct:>9.2f}%")
    
    print("-" * 70)
    
    # Error promedio y máximo
    error_promedio = np.mean(errores)
    error_maximo = np.max(errores)
    
    print(f"\n📉 ERROR PROMEDIO: {error_promedio:.2f}%")
    print(f"📉 ERROR MÁXIMO: {error_maximo:.2f}%")
    
    # Interpretación
    print("\n🎯 INTERPRETACIÓN:")
    if error_promedio < 1:
        print("✅ EXCELENTE: La calidad del texto se preserva prácticamente igual")
    elif error_promedio < 3:
        print("✅ MUY BUENO: La calidad del texto es muy representativa")
    elif error_promedio < 5:
        print("⚠️  ACEPTABLE: Hay pequeñas diferencias en calidad")
    else:
        print("❌ PROBLEMA: El downsampling afecta significativamente la calidad")



COMPARACIÓN Y CÁLCULO DE ERRORES

📊 COMPARACIÓN DE MÉTRICAS:
----------------------------------------------------------------------
Métrica                                    Original     Downsampled    Error %
----------------------------------------------------------------------
promedio_chars                               504.25          503.51      0.15%
mediana_chars                                403.00          403.00      0.00%
std_chars                                    418.55          417.91      0.15%
p25_chars                                    198.00          198.00      0.00%
p75_chars                                    666.00          665.00      0.15%
p95_chars                                   1423.00         1421.00      0.14%
promedio_palabras                             89.36           89.23      0.14%
mediana_palabras                              70.00           70.00      0.00%
longitud_palabra_promedio                      4.48            4.48      0.01%
------

In [11]:
# PASO 5: Análisis detallado de vocabulario
print("\n" + "="*70)
print("ANÁLISIS DE VOCABULARIO")
print("="*70)

if stats_original and stats_downsampled:
    print(f"\n📚 VOCABULARIO ÚNICO:")
    print(f"  Original: {stats_original['vocabulario_unico']:,} palabras únicas")
    print(f"  Downsampled: {stats_downsampled['vocabulario_unico']:,} palabras únicas")
    
    ratio_vocab = stats_downsampled['vocabulario_unico'] / stats_original['vocabulario_unico']
    print(f"  Ratio: {ratio_vocab:.2%} del vocabulario preservado")
    
    print(f"\n📝 TOTAL DE PALABRAS:")
    print(f"  Original: {stats_original['total_palabras']:,} palabras totales")
    print(f"  Downsampled: {stats_downsampled['total_palabras']:,} palabras totales")
    
    print(f"\n🔤 PALABRAS MÁS FRECUENTES (Original):")
    for palabra, frecuencia in stats_original['palabras_mas_frecuentes']:
        print(f"  {palabra:<20} {frecuencia:,}")
    
    print(f"\n🔤 PALABRAS MÁS FRECUENTES (Downsampled):")
    for palabra, frecuencia in stats_downsampled['palabras_mas_frecuentes']:
        print(f"  {palabra:<20} {frecuencia:,}")



ANÁLISIS DE VOCABULARIO

📚 VOCABULARIO ÚNICO:
  Original: 160,581 palabras únicas
  Downsampled: 90,915 palabras únicas
  Ratio: 56.62% del vocabulario preservado

📝 TOTAL DE PALABRAS:
  Original: 180,501,866 palabras totales
  Downsampled: 45,007,774 palabras totales

🔤 PALABRAS MÁS FRECUENTES (Original):
  the                  14,272,343
  to                   5,463,523
  and                  5,165,930
  a                    3,887,164
  was                  3,746,104
  i                    3,535,604
  of                   2,553,744
  it                   2,441,960
  vehicle              2,225,455
  on                   2,219,902

🔤 PALABRAS MÁS FRECUENTES (Downsampled):
  the                  3,554,891
  to                   1,361,170
  and                  1,288,872
  a                    969,642
  was                  931,894
  i                    882,706
  of                   636,382
  it                   609,754
  on                   554,585
  vehicle              553,890


In [12]:
# PASO 6: Guardar resultados
import json

resultados = {
    'stats_original': stats_original,
    'stats_downsampled': stats_downsampled,
    'n_registros_original': len(df_filtered),
    'n_registros_downsampled': len(df_downsampled)
}

# Guardar (comando para copiar manualmente por compatibilidad JSON)
print("\n" + "="*70)
print("RESUMEN FINAL")
print("="*70)

if stats_original and stats_downsampled:
    print(f"\n✅ LONGITUD DEL TEXTO:")
    print(f"  Original:")
    print(f"    Promedio: {stats_original['promedio_chars']:.0f} caracteres")
    print(f"    Mediana: {stats_original['mediana_chars']:.0f} caracteres")
    print(f"    P25: {stats_original['p25_chars']:.0f}, P75: {stats_original['p75_chars']:.0f}, P95: {stats_original['p95_chars']:.0f}")
    
    print(f"\n  Downsampled (25%):")
    print(f"    Promedio: {stats_downsampled['promedio_chars']:.0f} caracteres")
    print(f"    Mediana: {stats_downsampled['mediana_chars']:.0f} caracteres")
    print(f"    P25: {stats_downsampled['p25_chars']:.0f}, P75: {stats_downsampled['p75_chars']:.0f}, P95: {stats_downsampled['p95_chars']:.0f}")
    
    print(f"\n✅ PALABRAS:")
    print(f"  Original:")
    print(f"    Promedio: {stats_original['promedio_palabras']:.0f} palabras")
    print(f"    Mediana: {stats_original['mediana_palabras']:.0f} palabras")
    
    print(f"\n  Downsampled (25%):")
    print(f"    Promedio: {stats_downsampled['promedio_palabras']:.0f} palabras")
    print(f"    Mediana: {stats_downsampled['mediana_palabras']:.0f} palabras")
    
    print(f"\n✅ CONCLUSIÓN:")
    error_promedio = np.mean([
        abs((stats_original['promedio_chars'] - stats_downsampled['promedio_chars']) / stats_original['promedio_chars']) * 100,
        abs((stats_original['mediana_chars'] - stats_downsampled['mediana_chars']) / stats_original['mediana_chars']) * 100,
        abs((stats_original['promedio_palabras'] - stats_downsampled['promedio_palabras']) / stats_original['promedio_palabras']) * 100,
        abs((stats_original['mediana_palabras'] - stats_downsampled['mediana_palabras']) / stats_original['mediana_palabras']) * 100
    ])
    
    print(f"  El downsampling (25%) preserva la calidad del texto con un error promedio de {error_promedio:.2f}%")



RESUMEN FINAL

✅ LONGITUD DEL TEXTO:
  Original:
    Promedio: 504 caracteres
    Mediana: 403 caracteres
    P25: 198, P75: 666, P95: 1423

  Downsampled (25%):
    Promedio: 504 caracteres
    Mediana: 403 caracteres
    P25: 198, P75: 665, P95: 1421

✅ PALABRAS:
  Original:
    Promedio: 89 palabras
    Mediana: 70 palabras

  Downsampled (25%):
    Promedio: 89 palabras
    Mediana: 70 palabras

✅ CONCLUSIÓN:
  El downsampling (25%) preserva la calidad del texto con un error promedio de 0.07%
